In [1]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

# Load preprocessed data
X_train = np.load('../data/X_train.npy')
X_test = np.load('../data/X_test.npy')
y_train = np.load('../data/y_train.npy')
y_test = np.load('../data/y_test.npy')

# Convert to PyTorch tensors
X_train = torch.tensor(X_train)
X_test = torch.tensor(X_test)
y_train = torch.tensor(y_train, dtype=torch.long)
y_test = torch.tensor(y_test, dtype=torch.long)

print("X_train:", X_train.shape)
print("Ready for training!")

X_train: torch.Size([176000, 2, 128])
Ready for training!


In [2]:
class SignalCNN(nn.Module):
    def __init__(self, num_classes=11):
        super(SignalCNN, self).__init__()
        
        self.conv_block = nn.Sequential(
            nn.Conv1d(2, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv1d(64, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Dropout(0.5)
        )
        
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 64, 128),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(128, num_classes)
        )
    
    def forward(self, x):
        x = self.conv_block(x)
        x = self.classifier(x)
        return x

model = SignalCNN(num_classes=11)
print(model)

SignalCNN(
  (conv_block): Sequential(
    (0): Conv1d(2, 64, kernel_size=(3,), stride=(1,), padding=(1,))
    (1): ReLU()
    (2): Conv1d(64, 64, kernel_size=(3,), stride=(1,), padding=(1,))
    (3): ReLU()
    (4): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (5): Dropout(p=0.5, inplace=False)
  )
  (classifier): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=4096, out_features=128, bias=True)
    (2): ReLU()
    (3): Dropout(p=0.5, inplace=False)
    (4): Linear(in_features=128, out_features=11, bias=True)
  )
)


In [3]:
# DataLoaders
train_dataset = TensorDataset(X_train, y_train)
test_dataset = TensorDataset(X_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=256, shuffle=False)

# Loss and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

print("Ready to train!")

Ready to train!


In [4]:
num_epochs = 10

for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    
    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch+1}/{num_epochs} - Loss: {avg_loss:.4f}")

print("Training done!")

Epoch 1/10 - Loss: 2.3982
Epoch 2/10 - Loss: 2.3300
Epoch 3/10 - Loss: 2.1723
Epoch 4/10 - Loss: 2.0560
Epoch 5/10 - Loss: 2.0009
Epoch 6/10 - Loss: 1.9563
Epoch 7/10 - Loss: 1.9316
Epoch 8/10 - Loss: 1.9140
Epoch 9/10 - Loss: 1.9025
Epoch 10/10 - Loss: 1.8917
Training done!


In [5]:
model.eval()
correct = 0
total = 0

with torch.no_grad():
    for X_batch, y_batch in test_loader:
        outputs = model(X_batch)
        _, predicted = torch.max(outputs, 1)
        total += y_batch.size(0)
        correct += (predicted == y_batch).sum().item()

accuracy = 100 * correct / total
print(f"Test Accuracy: {accuracy:.2f}%")

Test Accuracy: 33.95%


In [6]:
for epoch in range(10):
    model.train()
    total_loss = 0
    
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    
    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch+11}/20 - Loss: {avg_loss:.4f}")

print("Done!")

Epoch 11/20 - Loss: 1.8772
Epoch 12/20 - Loss: 1.8610
Epoch 13/20 - Loss: 1.8407
Epoch 14/20 - Loss: 1.8228
Epoch 15/20 - Loss: 1.8096
Epoch 16/20 - Loss: 1.7973
Epoch 17/20 - Loss: 1.7881
Epoch 18/20 - Loss: 1.7730
Epoch 19/20 - Loss: 1.7573
Epoch 20/20 - Loss: 1.7309
Done!


In [7]:
model.eval()
correct = 0
total = 0

with torch.no_grad():
    for X_batch, y_batch in test_loader:
        outputs = model(X_batch)
        _, predicted = torch.max(outputs, 1)
        total += y_batch.size(0)
        correct += (predicted == y_batch).sum().item()

accuracy = 100 * correct / total
print(f"Test Accuracy: {accuracy:.2f}%")

Test Accuracy: 40.32%


In [8]:
torch.save(model.state_dict(), '../models/signal_cnn.pth')
print("Model saved!")

Model saved!


In [9]:
for epoch in range(10):
    model.train()
    total_loss = 0
    
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    
    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch+21}/30 - Loss: {avg_loss:.4f}")

print("Done!")

Epoch 21/30 - Loss: 1.7047
Epoch 22/30 - Loss: 1.6789
Epoch 23/30 - Loss: 1.6587
Epoch 24/30 - Loss: 1.6461
Epoch 25/30 - Loss: 1.6343
Epoch 26/30 - Loss: 1.6217
Epoch 27/30 - Loss: 1.6104
Epoch 28/30 - Loss: 1.5936
Epoch 29/30 - Loss: 1.5807
Epoch 30/30 - Loss: 1.5727
Done!


In [10]:
model.eval()
correct = 0
total = 0

with torch.no_grad():
    for X_batch, y_batch in test_loader:
        outputs = model(X_batch)
        _, predicted = torch.max(outputs, 1)
        total += y_batch.size(0)
        correct += (predicted == y_batch).sum().item()

accuracy = 100 * correct / total
print(f"Test Accuracy: {accuracy:.2f}%")

Test Accuracy: 44.10%


In [11]:
torch.save(model.state_dict(), '../models/signal_cnn.pth')
print("Model saved!")

Model saved!


In [ ]:
import pickle
import numpy as np

with open('../data/RML2016.10a_dict.pkl', 'rb') as f:
    data = pickle.load(f, encoding='latin1')

modulations = sorted(list(set([k[0] for k in data.keys()])))
snrs = sorted(list(set([k[1] for k in data.keys()])))
snr_accuracy = {}

model.eval()
with torch.no_grad():
    for snr in snrs: # loop through each SNR level (-20 to +18)
        X_snr = []
        y_snr = []
        
        for mod in modulations: # for each SNR, collect all 11 modulations
            signals = data[(mod, snr)] # get the 1000 signals for this combo
            for signal in signals:
                X_snr.append(signal)
                y_snr.append(modulations.index(mod)) # convert mod name to number
        
        X_snr = torch.tensor(np.array(X_snr, dtype=np.float32))
        y_snr = torch.tensor(np.array(y_snr))
        
        outputs = model(X_snr)
        _, predicted = torch.max(outputs, 1)
        acc = (predicted == y_snr).float().mean().item() * 100
        snr_accuracy[snr] = acc
        print(f"SNR {snr:4d} dB: {acc:.1f}%")

SNR  -20 dB: 9.2%
SNR  -18 dB: 9.3%
SNR  -16 dB: 9.5%
SNR  -14 dB: 9.8%
SNR  -12 dB: 10.8%
SNR  -10 dB: 14.4%
SNR   -8 dB: 22.5%
SNR   -6 dB: 34.9%
SNR   -4 dB: 52.0%
SNR   -2 dB: 59.7%
SNR    0 dB: 63.3%
SNR    2 dB: 65.5%
SNR    4 dB: 66.4%
SNR    6 dB: 66.3%
SNR    8 dB: 68.1%
SNR   10 dB: 68.0%
SNR   12 dB: 67.7%
SNR   14 dB: 68.3%
SNR   16 dB: 66.0%
SNR   18 dB: 67.1%
